In [ ]:
import warnings
from functools import partial

import iris
import matplotlib.colors as mcol
import matplotlib.pyplot as plt
import numpy as np
import paths
from aeolus.calc import (
    integrate,
    last_n_day_mean,
    spatial,
    spatial_mean,
    time_mean,
    toa_net_energy,
    zonal_mean,
)
from aeolus.const import init_const
from aeolus.coord import (
    get_cube_rel_days,
    interp_cube_from_height_to_pressure_levels,
    isel,
)
from aeolus.lfric import fix_time_coord, load_lfric_raw
from aeolus.model import lfric, um
from aeolus.plot import (
    figsave,
    hcross,
    map_scatter,
    stream,
    timeseries_1d,
    timeseries_2d,
)
from common import (
    CATEGORIES,
    EXPERIMENTS,
    lfric_callback_uniform_height,
)

CNORM = {"norm": mcol.CenteredNorm(vcenter=0), "cmap": "RdBu_r"}

warnings.filterwarnings("ignore")
iris.FUTURE.date_microseconds = True

lfric.w = "w_in_w3"
lfric.dt_force = "temperature_increment_from_external_forcing"

print(*EXPERIMENTS.keys(), sep="\n")

In [ ]:
# f = init_const().stefan_boltzmann * iris.cube.Cube(300, units="K") ** 4
# f.convert_units("W m-2")
# f, f.data
# exp_key = "hd209_uniform_t100k_c48"
# ds_r = load_lfric_raw(
#     sorted(
#         (
#             paths.data_work
#             / "lfric"
#             / EXPERIMENTS[exp_key].group
#             / EXPERIMENTS[exp_key].label
#         ).glob("restart*.nc")
#     )[:4],
# )

# fig, axs = plt.subplots(nrows=2, ncols=2)
# for i in range(4):
#     map_scatter(ds_r.extract("internal_flux")[i], s=0.1, ax=axs.flat[i])

In [ ]:
# exp_key = "shj_c48_l32"

# ds_ll = load_lfric_raw(
#     sorted(
#         (
#             paths.data_work
#             / "lfric"
#             / EXPERIMENTS[exp_key].group
#             / EXPERIMENTS[exp_key].label
#         ).glob("lfric_diag_regr*.nc")
#     ),
#     callback=partial(
#         lfric_callback_uniform_height,
#         model_top_height=EXPERIMENTS[exp_key].const.domain_height.data,
#     ),
# )
# print(ds_ll)

ds_ll = load_lfric_raw(
    sorted(
        (paths.data_work / "lfric" / "camembert" / "case2_gj1214b").glob(
            "lfric_diag_latlon*.nc"
        )
    ),
    callback=partial(
        lfric_callback_uniform_height,
        model_top_height=4e6,
    ),
)
print(ds_ll)

In [ ]:
plt.plot(zonal_mean(ds_ll.extract_cube("air_pressure")[0, 0, ...]).data * 1e-2)
plt.plot(zonal_mean(ds_ll.extract_cube("air_pressure")[-1, 0, ...]).data * 1e-2)

In [ ]:
hcross(ds_ll.extract_cube(lfric.u)[-1, 64, ...])

In [ ]:
plt.pcolormesh(
    ds_ll.extract_cube("longwave_heating_rate")[0, :, :, 72].data
    # +
    # ds_ll.extract_cube("shortwave_heating_rate")[
    #         0, :, :, 72
    #     ].data
)
plt.colorbar()

In [ ]:
p = spatial_mean(ds_ll.extract_cube("air_pressure")[0, :, :, :]).data
z = ds_ll.extract_cube("air_pressure").coord("level_height").points

In [ ]:
ax = plt.axes()
ax.plot(
    ds_ll.extract_cube("potential_temperature_increment_from_slow_physics")[
        0, :25, 15, 0
    ].data,
    np.log10(p[:25]),
)
ax.invert_yaxis()

In [ ]:
ax = plt.axes()
ax.plot(p[:20] / 1e5, z[:20])
ax.set_xscale("log")
ax.axvline(10, color="k")
ax.axvline(100, color="k")
ax.axvline(1, color="k")
ax.axhline((-z / np.log(p / p[0])).mean(), color="k")
ax.axhline((-2 * z / np.log(p / p[0])).mean(), color="k")
ax.axhline((-3 * z / np.log(p / p[0])).mean(), color="k")
ax.axhline((-4 * z / np.log(p / p[0])).mean(), color="k")

In [ ]:
plt.pcolormesh(
    ds_ll.extract_cube("potential_temperature_increment_from_slow_physics")[
        -1, 15, :, :
    ].data
)
plt.colorbar()

In [ ]:
diag_key = "u"
target_pressure = 7500 * 1e-3
n_days_mean = 300
target_sim_days = 10_000
var = ds_ll.extract_cube(getattr(lfric, diag_key))
u = ds_ll.extract_cube(lfric.u)
v = ds_ll.extract_cube(lfric.v)
method_plt = "pcolormesh"
n_contours = 7
# kw_plt = {"levels": np.arange(-15_000, 15_001, 1000), "cmap": "RdBu_r", "rasterized": True}
# norm = mcol.TwoSlopeNorm(vcenter=0, vmin=-1000, vmax=9_000)
norm = mcol.TwoSlopeNorm(vcenter=0)
# kw_plt = {"vmin": -12000, "vmax": 12000, "cmap": "RdBu_r", "rasterized": True}
kw_plt = {"norm": norm, "cmap": "RdBu_r", "rasterized": True}
pres = ds_ll.extract_cube(lfric.pres).interpolate(
    [(lfric.z, var.coord(lfric.z).points)], iris.analysis.Linear()
)
var_plev = interp_cube_from_height_to_pressure_levels(var, pres, [target_pressure])
u_plev = interp_cube_from_height_to_pressure_levels(u, pres, [target_pressure])
v_plev = interp_cube_from_height_to_pressure_levels(v, pres, [target_pressure])
pres_equator = iris.util.squeeze(
    last_n_day_mean(
        zonal_mean(pres.interpolate([(lfric.y, [0])], iris.analysis.Linear())),
        n_days_mean,
    )
)

fig = plt.figure(figsize=(6, 4), layout="constrained")
axd = fig.subplot_mosaic(
    [["z-y", "t-z", "cbar"], ["y-x", "t-y", "cbar"]],
    gridspec_kw={"width_ratios": [1, 1, 0.02]},
)

ax = axd["z-y"]
ax.set_title(f"{n_days_mean:.0f}-day and zonal mean")
ax.set_xlabel("Latitude [deg]")
ax.set_xlim(-90, 90)
ax.set_ylabel("Pressure [bar]")
ax.invert_yaxis()
ax.set_yscale("log")
im = getattr(ax, method_plt)(
    var.coord(lfric.y).points,
    pres_equator.data * 1e-5,
    last_n_day_mean(zonal_mean(var), n_days_mean).data,
    **kw_plt,
)
if n_contours:
    cs = ax.contour(
        var.coord(lfric.y).points,
        pres_equator.data * 1e-5,
        last_n_day_mean(zonal_mean(var), n_days_mean).data,
        n_contours,
        # levels=np.arange(-20_000, 20_001, contour_step),
        colors="k",
        linewidths=0.5,
    )
    ax.clabel(cs, fmt="%d", fontsize="xx-small")
ax.axhline(target_pressure * 1e-5, color="k", lw=1, ls="--", dash_capstyle="round")

ax = axd["t-z"]
ax.set_title("Zonal mean, equator")
ax.set_ylabel("Pressure [bar]")
ax.invert_yaxis()
ax.set_yscale("log")
ax.set_xlabel("Time [days]")
ax.set_xlim(0, target_sim_days)
im = getattr(ax, method_plt)(
    get_cube_rel_days(var),
    pres_equator.data * 1e-5,
    iris.util.squeeze(
        zonal_mean(var.interpolate([(lfric.y, [0])], iris.analysis.Linear()))
    ).data.T,
    **kw_plt,
)
ax.axhline(target_pressure * 1e-5, color="k", lw=1, ls="--", dash_capstyle="round")
ax.axvspan(
    0,
    target_sim_days - n_days_mean,
    color="k",
    alpha=0.05,
)

ax = axd["y-x"]
ax.set_title(f"{n_days_mean:.0f}-day mean, {target_pressure * 1e-5:>5.2f} bar")
ax.set_ylabel("Latitude [deg]")
ax.set_ylim(-90, 90)
ax.set_xlabel("Longitude [deg]")
ax.set_xlim(0, 360)
im = getattr(ax, method_plt)(
    var.coord(lfric.x).points,
    var.coord(lfric.y).points,
    last_n_day_mean(var_plev, n_days_mean).data,
    **kw_plt,
)
stream(
    last_n_day_mean(u_plev, n_days_mean),
    last_n_day_mean(v_plev, n_days_mean),
    color="k",
    density=0.75,
    arrowsize=0.5,
    linewidth=0.5,
    ax=ax,
)

ax = axd["t-y"]
ax.set_title(f"Zonal mean, {target_pressure * 1e-5:>5.2f} bar")
ax.set_ylabel("Latitude [deg]")
ax.set_ylim(-90, 90)
ax.set_xlabel("Time [days]")
ax.set_xlim(0, target_sim_days)
im = getattr(ax, method_plt)(
    get_cube_rel_days(var),
    var.coord(lfric.y).points,
    zonal_mean(var_plev).data.T,
    **kw_plt,
)
ax.axvspan(
    0,
    target_sim_days - n_days_mean,
    color="k",
    alpha=0.05,
)

fig.colorbar(im, cax=axd["cbar"], orientation="vertical")
# fig.suptitle(
#     "\n".join([
#         f"{EXPERIMENTS[exp_key].title} | LFRic-Atmosphere with {CATEGORIES[EXPERIMENTS[exp_key].category].title}",
#         f"{var.name()} / {var.units}",
#     ])
# )

# figsave(
#     fig,
#     paths.drafts / exp_key / f"{exp_key}_{diag_key}_quicklook",
# )

In [ ]:
fig = plt.figure(figsize=(4, 3), layout="constrained")
axd = fig.subplot_mosaic(
    [["z-y", "cbar"], ["z-y", "cbar"]],
    gridspec_kw={"width_ratios": [1, 0.02]},
)

ax = axd["z-y"]
ax.set_title(f"Zonal Mean Eastward Wind [$m$ $s^{{{-1}}}$]")
ax.set_xlabel("Latitude [deg]")
ax.set_xlim(-90, 90)
ax.set_ylabel("Pressure [bar]")
ax.invert_yaxis()
ax.set_yscale("log")
im = getattr(ax, method_plt)(
    var.coord(lfric.y).points,
    pres_equator.data * 1e-5,
    last_n_day_mean(zonal_mean(var), n_days_mean).data,
    **kw_plt,
)
if n_contours:
    cs = ax.contour(
        var.coord(lfric.y).points,
        pres_equator.data * 1e-5,
        last_n_day_mean(zonal_mean(var), n_days_mean).data,
        n_contours,
        # levels=np.arange(-20_000, 20_001, contour_step),
        colors="k",
        linewidths=0.5,
    )
    ax.clabel(cs, fmt="%d", fontsize="xx-small")

fig.colorbar(im, cax=axd["cbar"], orientation="vertical")
fig.suptitle(
    "\n".join([
        f"{EXPERIMENTS[exp_key].title} | LFRic-Atmosphere with {CATEGORIES[EXPERIMENTS[exp_key].category].title}"
    ])
)

figsave(
    fig,
    paths.drafts / exp_key / f"{exp_key}_{diag_key}_zonal_mean",
)